### Forward Crossref funder matching: ROR, then names, FundRef DOI as backup

New references use unique ROR, then exact name/alias, hierarchy leaf, reviewed aliases, twins,
programmes. The FundRef DOI is a backup key (funder_fundref_owners: the registry's internal key
plus ROR's FundRef crosswalk; withdrawn/dropped funders never match): it fills references with no
name match and confirms a held name match to the same funder. On a DOI/name conflict the name
answer wins. Name holds do not apply to DOI matches. New or changed references resolved by ROR or
FundRef DOI carry their award ids; name-resolved references are links only (#1303).
The immutable reference cut excludes pre-deployment references. Published pairs
and their award ids survive, including historical redirected IDs (ids are added, never removed), except links
Kyle's trap rule distrusts (rule 1, crossref_trap_funders.csv): the evidence marks them
'distrusted' and they leave the junction.
The catalogue-bound lookup must be active before this reader runs.
Jev nm1012_jev_* residuals remain unwired; gated results could enter after the
deterministic ladder for unresolved references, with the same served/identity checks.


In [ ]:
%sql
-- Override with SET VAR only for an isolated dev replay.
DECLARE OR REPLACE VARIABLE funder_name_lookup STRING DEFAULT 'openalex.funders.funder_name_lookup';
DECLARE OR REPLACE VARIABLE funder_names_active BOOLEAN DEFAULT false;


In [ ]:
%sql
-- Resolve optional release at runtime. The absent relation is never analyzed.
-- The no-DOI activation assertion below fails closed if the lookup is absent/stale.
BEGIN
CREATE OR REPLACE TEMP VIEW funder_name_lookup_local AS
SELECT CAST(NULL AS STRING) rule,CAST(NULL AS STRING) name_key,CAST(NULL AS BIGINT) funder_id WHERE false;
IF (SELECT count(*) FROM system.information_schema.tables
 WHERE table_catalog=split_part(funder_name_lookup,'.',1)
 AND table_schema=split_part(funder_name_lookup,'.',2)
 AND table_name=split_part(funder_name_lookup,'.',3))=1 THEN
  EXECUTE IMMEDIATE 'CREATE OR REPLACE TEMP VIEW funder_name_lookup_local AS
    SELECT rule,name_key,funder_id FROM ' || funder_name_lookup || '
    WHERE catalogue_signature = (SELECT concat(count(*), char(58), cast(sum(cast(xxhash64(funder_id,display_name,alternate_titles,ror_id,doi,crossref_id,merge_into_id) AS DECIMAL(38,0))) AS STRING)) FROM openalex.funders.funders)';
END IF;
END;


In [ ]:
%sql
-- Top-level SET VAR (rejected inside BEGIN...END on the DBSQL warehouse). Same values as before.
SET VAR funder_names_active = (SELECT EXISTS(SELECT 1 FROM funder_name_lookup_local));
SET VAR funder_name_lookup = 'funder_name_lookup_local';


In [ ]:
%sql
SELECT assert_true(funder_names_active,'No-DOI matching requires the reviewed catalogue-bound name lookup; junction unchanged');


In [ ]:
%sql
-- FundRef DOI -> funder: a demoted backup join key (standup 10-15). Crossref now deposits ROR, but FundRef DOIs remain
-- in records ("if you find these, it means it's that funder"). Crossref: unique ROR, then names, with this key filling
-- references with no name match and confirming same-funder names (a disagreeing name wins). DataCite: ROR, this key, names.
-- Keys: (1) the registry's internal FundRef key as served by the signed catalogue (crossref_id, merge losers project to
-- their survivor) and (2) ROR's FundRef crosswalk (active ROR records only) for ids no catalogue row carries, through a
-- uniquely owned ROR id. Never resolves: a key with more than one owner, a merged or unserved target, and any id or
-- funder in openalex.funders.deleted_funders (withdrawn and dropped funders, e.g. batch dd1016-b3).
CREATE OR REPLACE TEMP VIEW funder_fundref_owners AS
WITH deleted AS (
 SELECT funder_id,CAST(crossref_id AS STRING) fk FROM openalex.funders.deleted_funders
),
catalogue_keys AS (
 SELECT DISTINCT CAST(crossref_id AS STRING) fk,coalesce(merge_into_id,funder_id) funder_id,'catalogue' key_source
 FROM openalex.funders.funders WHERE crossref_id IS NOT NULL
),
ror_owners AS (
 SELECT lower(regexp_extract(trim(ror_id),r'(?i)^(?:https?://ror\.org/)?([0-9a-z]{9})/?$',1)) k,
 count(DISTINCT coalesce(merge_into_id,funder_id)) owners,max(coalesce(merge_into_id,funder_id)) funder_id
 FROM openalex.funders.funders GROUP BY 1
),
crosswalk_keys AS (
 SELECT DISTINCT x.fk,o.funder_id,'ror_crosswalk' key_source
 FROM (SELECT lower(regexp_extract(trim(r.id),r'(?i)^(?:https?://ror\.org/)?([0-9a-z]{9})/?$',1)) rid,
        regexp_extract(trim(v),r'(?i)^(?:(?:https?://(?:dx\.)?doi\.org/|doi:\s*)?10\.13039/)?([0-9]+)$',1) fk
       FROM openalex.institutions.ror r
       LATERAL VIEW explode(flatten(transform(filter(coalesce(r.external_ids,array()),e -> lower(e.type)='fundref'),e -> coalesce(e.all,array())))) a AS v
       WHERE r.status='active') x
 JOIN ror_owners o ON o.k=x.rid AND o.k<>'' AND o.owners=1
 LEFT ANTI JOIN catalogue_keys c ON c.fk=x.fk
 WHERE x.fk<>''
),
keys AS (
 SELECT k.* FROM (SELECT * FROM catalogue_keys UNION ALL SELECT * FROM crosswalk_keys) k
 LEFT ANTI JOIN deleted d ON d.fk=k.fk
 LEFT ANTI JOIN deleted g ON g.funder_id=k.funder_id
),
owners AS (
 SELECT fk,count(DISTINCT funder_id) owners,max(funder_id) funder_id,max(key_source) key_source FROM keys GROUP BY fk
)
SELECT o.fk,o.owners,CASE WHEN o.owners=1 AND s.id IS NOT NULL THEN o.funder_id END funder_id,o.key_source
FROM owners o
LEFT JOIN openalex.funders.funders t ON t.funder_id=o.funder_id AND t.merge_into_id IS NULL
LEFT JOIN openalex.funders.funders_api s ON s.id=t.funder_id


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW crossref_funder_references AS
SELECT lm.work_id,lower(lm.native_id) native_id,f.name funder_name,f.ror funder_ror,f.awards award_ids,
 lower(regexp_extract(trim(f.ror),r'(?i)^(?:https?://ror\.org/)?([0-9a-z]{9})/?$',1)) rk,
 regexp_extract(trim(f.doi),r'(?i)^(?:(?:https?://(?:dx\.)?doi\.org/|doi:\s*)?10\.13039/)?([0-9]+)$',1) fk,
 sha2(to_json(named_struct('native_id',lower(lm.native_id),'name',f.name,'ror',f.ror,'awards',array_sort(f.awards))),256) reference_key
FROM openalex.works.locations_mapped lm
LATERAL VIEW explode(funders) e AS f
WHERE lm.provenance='crossref' AND lm.work_id IS NOT NULL


In [ ]:
%sql
-- Immutable deployment cut. Never replace this table during ordinary runs.
-- Initial execution freezes existing references; later new/changed content is forward input.
CREATE TABLE IF NOT EXISTS openalex.awards.crossref_funder_reference_cut
USING delta AS SELECT DISTINCT reference_key FROM crossref_funder_references


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW crossref_funder_name_candidates AS
WITH refs AS (SELECT r.* FROM crossref_funder_references r LEFT ANTI JOIN openalex.awards.crossref_funder_reference_cut c USING(reference_key)),
ror_owners AS (
 SELECT lower(regexp_extract(trim(ror_id),r'(?i)^(?:https?://ror\.org/)?([0-9a-z]{9})/?$',1)) k,
 count(DISTINCT coalesce(merge_into_id,funder_id)) owners,max(coalesce(merge_into_id,funder_id)) funder_id
 FROM openalex.funders.funders GROUP BY 1
),
name_keys AS (SELECT DISTINCT lower(trim(funder_name)) k FROM refs),
name_key_parts AS (SELECT *,trim(element_at(split(k,r'\|'),-1)) leaf FROM name_keys),
name_key_normalized AS (SELECT *,trim(regexp_replace(regexp_replace(regexp_replace(translate(lower(replace(replace(replace(leaf,'&amp;','&'),'&apos;',char(39)),'&#39;',char(39))), 'áàâäãåāăąçćčďđéèêëēėęěíìîïīıłñńňóòôöõøōőřśšşťúùûüūůűýÿžźż', 'aaaaaaaaacccddeeeeeeeeiiiiiilnnnoooooooorssstuuuuuuuyyzzz'), '&', ' and '), r'[^\p{L}\p{N}]+', ' '), r'(^| )the( |$)', ' ')) lk,trim(regexp_replace(regexp_replace(regexp_replace(translate(lower(replace(replace(replace(regexp_replace(leaf,r'\s*\([a-z0-9 .&-]{2,15}\)\s*$',''),'&amp;','&'),'&apos;',char(39)),'&#39;',char(39))), 'áàâäãåāăąçćčďđéèêëēėęěíìîïīıłñńňóòôöõøōőřśšşťúùûüūůűýÿžźż', 'aaaaaaaaacccddeeeeeeeeiiiiiilnnnoooooooorssstuuuuuuuyyzzz'), '&', ' and '), r'[^\p{L}\p{N}]+', ' '), r'(^| )the( |$)', ' ')) lsk FROM name_key_parts),
name_rules AS (
 SELECT k.k,coalesce(e.funder_id,h.funder_id,hs.funder_id,a.funder_id,t.funder_id,p.funder_id) funder_id, CASE WHEN e.funder_id IS NOT NULL THEN 'exact' WHEN coalesce(h.funder_id,hs.funder_id) IS NOT NULL THEN 'hierarchy' WHEN a.funder_id IS NOT NULL THEN 'reviewed' WHEN t.funder_id IS NOT NULL THEN 'twin' WHEN p.funder_id IS NOT NULL THEN 'programme' END name_rule
 FROM name_key_normalized k
 LEFT JOIN funder_name_lookup_local e ON e.rule='exact' AND e.name_key=k.k
 LEFT JOIN funder_name_lookup_local h ON h.rule='hierarchy' AND h.name_key=k.lk AND k.k LIKE '%|%'
 LEFT JOIN funder_name_lookup_local hs ON hs.rule='hierarchy' AND hs.name_key=k.lsk AND k.k LIKE '%|%'
 LEFT JOIN funder_name_lookup_local a ON a.rule='reviewed' AND a.name_key=k.k
 LEFT JOIN funder_name_lookup_local t ON t.rule='twin' AND t.name_key=k.k
 LEFT JOIN funder_name_lookup_local p ON p.rule='programme' AND p.name_key=k.k
),
conflict_name_variants AS (
 SELECT coalesce(merge_into_id,funder_id) canonical_id,lower(trim(nm)) k
 FROM openalex.funders.funders
 LATERAL VIEW explode(array_union(array(display_name),coalesce(from_json(alternate_titles,'array<string>'),array()))) n AS nm
), conflict_name_owners AS (
 SELECT k,count(DISTINCT canonical_id) owners FROM conflict_name_variants WHERE k<>'' GROUP BY k
), conflict_display_owners AS (
 SELECT lower(trim(display_name)) k,count(DISTINCT coalesce(merge_into_id,funder_id)) owners
 FROM openalex.funders.funders GROUP BY 1
),
reviewed_name_holds AS (
 SELECT cast(unhex(k) AS STRING) name_key,reason FROM VALUES
 ('3937332070726f6772616d','predecessor_programme_scope'),
 ('6162626f7474','reviewed_generic_ambiguity'),
 ('617263','reviewed_generic_ambiguity'),
 ('62617366','reviewed_generic_ambiguity'),
 ('62696a7a6f6e646572206f6e6465727a6f656b73666f6e6473','reviewed_generic_ambiguity'),
 ('627261696e20666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('63616e6f6e20666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('636173','reviewed_generic_ambiguity'),
 ('636561','reviewed_generic_ambiguity'),
 ('6368696e6120756e6976657273697479206f662067656f736369656e636573','reviewed_generic_ambiguity'),
 ('63736963','reviewed_generic_ambiguity'),
 ('63736972','reviewed_generic_ambiguity'),
 ('6465616e73686970206f6620736369656e7469666963207265736561726368','reviewed_generic_ambiguity'),
 ('64656c7461','reviewed_generic_ambiguity'),
 ('6465706172746d656e74206f662062696f746563686e6f6c6f6779','reviewed_generic_ambiguity'),
 ('6465706172746d656e74206f66206865616c7468','reviewed_generic_ambiguity'),
 ('6465706172746d656e74206f6620736369656e636520616e6420746563686e6f6c6f6779','reviewed_generic_ambiguity'),
 ('656d657267656e6379206d65646963696e6520666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('656e7669726f6e6d656e74616c2070726f74656374696f6e206167656e6379','reviewed_generic_ambiguity'),
 ('657361','reviewed_generic_ambiguity'),
 ('65736f','reviewed_generic_ambiguity'),
 ('6575','reviewed_generic_ambiguity'),
 ('6665646572','reviewed_generic_ambiguity'),
 ('666967687420666f72207369676874','reviewed_generic_ambiguity'),
 ('666f6e6465637974','reviewed_generic_ambiguity'),
 ('666f6f6420616e6420647275672061646d696e697374726174696f6e','reviewed_generic_ambiguity'),
 ('666f756e646174696f6e206669676874696e6720626c696e646e657373','reviewed_generic_ambiguity'),
 ('666f756e646174696f6e20666f7220736369656e636520616e6420746563686e6f6c6f6779','reviewed_generic_ambiguity'),
 ('66756e6420666f7220736369656e7469666963207265736561726368','reviewed_generic_ambiguity'),
 ('676f7665726e6d656e74206f6620737061696e206d696e6973747279206f662065636f6e6f6d7920616e6420636f6d70657469746976656e657373','reviewed_generic_ambiguity'),
 ('67736b','reviewed_generic_ambiguity'),
 ('6865616c746820666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('686561727420666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('68657263756c657320666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('696e6166','reviewed_generic_ambiguity'),
 ('696e666e','reviewed_generic_ambiguity'),
 ('696e6e6f766174696f6e2066756e64','reviewed_generic_ambiguity'),
 ('696e7374697475746520666f7220616476616e636564207374756479','reviewed_generic_ambiguity'),
 ('696e73746974757465206f662063616e636572207265736561726368','reviewed_generic_ambiguity'),
 ('696e74657266616365','reviewed_generic_ambiguity'),
 ('6a647266','reviewed_generic_ambiguity'),
 ('6a7576656e696c6520646961626574657320726573656172636820666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('6b6e6f776c6564676520666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('6b6f72656120676f7665726e6d656e74','reviewed_generic_ambiguity'),
 ('6b6f7265616e20676f7665726e6d656e74','reviewed_generic_ambiguity'),
 ('6d65646963616c20726573656172636820666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('6d6572636b','reviewed_generic_ambiguity'),
 ('6d657374','reviewed_generic_ambiguity'),
 ('6d696e65636f','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f66206167726963756c74757265','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f66206167726963756c7475726520616e6420727572616c20646576656c6f706d656e74','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620646566656e6365','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f662065636f6e6f6d69632061666661697273','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620656475636174696f6e','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620656475636174696f6e20616e6420736369656e6365','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620656475636174696f6e206d616c6179736961','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620656475636174696f6e2c20796f75746820616e642073706f727473','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620656e7669726f6e6d656e74616c2070726f74656374696f6e','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f662066696e616e6365','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f66206865616c7468','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f66206865616c746820616e642077656c66617265','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f662068696768657220656475636174696f6e','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f662068696768657220656475636174696f6e20616e6420736369656e7469666963207265736561726368','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f66206e61747572616c207265736f7572636573','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620726573656172636820616e6420696e6e6f766174696f6e','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620736369656e636520616e642068696768657220656475636174696f6e','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f6620736369656e636520616e6420746563686e6f6c6f6779','reviewed_generic_ambiguity'),
 ('6d696e6973747279206f662074686520656e7669726f6e6d656e74','reviewed_generic_ambiguity'),
 ('6d6f7374','reviewed_generic_ambiguity'),
 ('6d7364','reviewed_generic_ambiguity'),
 ('6e6174696f6e616c206167656e637920666f7220726573656172636820616e6420646576656c6f706d656e74','country_ambiguous_alias'),
 ('6e6174696f6e616c2062617369632072657365617263682070726f6772616d','predecessor_programme_scope'),
 ('6e6174696f6e616c2062617369632072657365617263682070726f6772616d206f66206368696e61','predecessor_programme_scope'),
 ('6e6174696f6e616c2062617369632072657365617263682070726f6772616d206f66206368696e6120283937332070726f6772616d29','predecessor_programme_scope'),
 ('6e6174696f6e616c2063616e6365722063656e746572','reviewed_generic_ambiguity'),
 ('6e6174696f6e616c20696e73746974757465206f66206865616c7468','reviewed_generic_ambiguity'),
 ('6e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('6e6174696f6e616c2070726f6772616d206f6e206b65792062617369632072657365617263682070726f6a65637420283937332070726f6772616d29','predecessor_programme_scope'),
 ('6e6174696f6e616c20726573656172636820636f756e63696c','reviewed_generic_ambiguity'),
 ('6e6174696f6e616c20726573656172636820666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('6e6174696f6e616c20726573656172636820666f756e646174696f6e20286e726629','reviewed_generic_ambiguity'),
 ('6e61747572616c20736369656e636520666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('6e617475726520657420746563686e6f6c6f67696573','reviewed_generic_ambiguity'),
 ('6e6970706f6e20666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('6e6f7274686561737465726e20756e6976657273697479','reviewed_generic_ambiguity'),
 ('6e6f7661','reviewed_generic_ambiguity'),
 ('6e7263','reviewed_generic_ambiguity'),
 ('6e7266','reviewed_generic_ambiguity'),
 ('6f6666696365206f6620746865206469726563746f72','reviewed_generic_ambiguity'),
 ('70617468','reviewed_generic_ambiguity'),
 ('726573656172636820616e6420646576656c6f706d656e74','reviewed_generic_ambiguity'),
 ('726f636b6574','reviewed_generic_ambiguity'),
 ('73616e74c3a9','reviewed_generic_ambiguity'),
 ('7363686f6f6c206f66206d65646963696e65','reviewed_generic_ambiguity'),
 ('736369656e636520616e6420746563686e6f6c6f677920646576656c6f706d656e742066756e64','reviewed_generic_ambiguity'),
 ('7368656c6c','reviewed_generic_ambiguity'),
 ('7374726f6b65206173736f63696174696f6e','reviewed_generic_ambiguity'),
 ('746563686e6f6c6f677920646576656c6f706d656e74','reviewed_generic_ambiguity'),
 ('746865206e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e','reviewed_generic_ambiguity'),
 ('74686520726573656172636820636f756e63696c','reviewed_generic_ambiguity'),
 ('756362','reviewed_generic_ambiguity'),
 ('756763','reviewed_generic_ambiguity'),
 ('756e616d','reviewed_generic_ambiguity'),
 ('756e697665727369646164206465206c6f7320616e646573','reviewed_generic_ambiguity'),
 ('756e6976657273697479206f66206b757264697374616e','reviewed_generic_ambiguity')
 h(k,reason)
),
global_ror_name_owners AS (
 -- Ownership only: do not use global ROR names to assign a new funder.
 SELECT lower(trim(n.value)) k,count(DISTINCT x.id) owners
 FROM openalex.institutions.ror x LATERAL VIEW explode(x.names) e AS n
 WHERE trim(n.value)<>'' GROUP BY 1
),
proposed AS (
 -- Precedence: unique ROR, then names; the FundRef DOI (backup key) fills references with no name match and confirms same-funder names.
 SELECT e.*,r.owners ror_owners,d.owners fundref_owners,d.funder_id fundref_funder_id,n.funder_id name_funder_id,
 coalesce(CASE WHEN r.owners=1 THEN r.funder_id END,CASE WHEN n.funder_id IS NULL OR n.funder_id=d.funder_id THEN d.funder_id END,n.funder_id) funder_id,
 CASE WHEN r.owners=1 THEN 'ROR' WHEN d.funder_id IS NOT NULL AND (n.funder_id IS NULL OR n.funder_id=d.funder_id) THEN 'FundRef DOI' ELSE n.name_rule END resolution_path
 FROM refs e
 LEFT JOIN ror_owners r ON r.k=e.rk AND r.k<>''
 LEFT JOIN funder_fundref_owners d ON d.fk=e.fk AND e.fk<>''
 LEFT JOIN name_rules n ON n.k=lower(trim(e.funder_name))
)
SELECT p.*, CASE WHEN p.resolution_path NOT IN ('ROR','FundRef DOI') AND h.reason IS NOT NULL THEN h.reason
 WHEN p.resolution_path='exact' AND lower(trim(p.funder_name)) RLIKE '^(research|development|technology|science|innovation|basic|applied|and|for|of|the|fund|funding|support|program|programme|project|projects)( (research|development|technology|science|innovation|basic|applied|and|for|of|the|fund|funding|support|program|programme|project|projects))*$' THEN 'generic_activity_name'
 WHEN p.resolution_path='exact' AND gr.owners>1 THEN 'global_name_ambiguity'
 WHEN p.resolution_path='exact' AND coalesce(trim(target.ror_id),'')='' AND NOT(trim(p.funder_name) RLIKE r'\s') THEN 'undocumented_short_name'
 WHEN p.resolution_path<>'ROR' AND p.resolution_path='exact' AND lower(trim(p.funder_name))<>lower(trim(target.display_name)) AND tr.owners>1 THEN 'target_ror_alias_scope'
 WHEN p.resolution_path NOT IN ('ROR','FundRef DOI') AND (n.owners>1 OR t.owners>1) THEN 'duplicate_name_identity' END hold_reason
FROM proposed p
JOIN openalex.funders.funders target ON target.funder_id=p.funder_id AND target.merge_into_id IS NULL
JOIN openalex.funders.funders_api served ON served.id=target.funder_id
LEFT JOIN conflict_name_owners n ON n.k=lower(trim(p.funder_name))
LEFT JOIN conflict_display_owners t ON t.k=lower(trim(target.display_name))
LEFT JOIN reviewed_name_holds h ON h.name_key=lower(trim(p.funder_name))
LEFT JOIN ror_owners tr ON tr.k=lower(regexp_extract(trim(target.ror_id),r'(?i)^(?:https?://ror\.org/)?([0-9a-z]{9})/?$',1)) AND tr.k<>''
LEFT JOIN global_ror_name_owners gr ON gr.k=lower(trim(p.funder_name))


In [ ]:
%sql
SELECT coalesce(hold_reason,'eligible') disposition,count(*) references,
 count(DISTINCT struct(work_id,funder_id)) pairs
FROM crossref_funder_name_candidates GROUP BY hold_reason


In [ ]:
%sql
-- FundRef DOI telemetry (informational). doi_name_conflicts: the DOI named another funder than the name ladder; the name answer wins.
SELECT resolution_path,coalesce(hold_reason,'eligible') disposition,count(*) references,
 count(DISTINCT struct(work_id,funder_id)) pairs,
 count_if(resolution_path<>'ROR' AND name_funder_id IS NOT NULL AND fundref_funder_id IS NOT NULL AND name_funder_id<>fundref_funder_id) doi_name_conflicts
FROM crossref_funder_name_candidates GROUP BY 1,2


In [ ]:
%sql
-- Materialize before replacing the junction. Published award ids are never removed.
-- The immutable main cut is staged once before deployment/merges, never recomputed here.
-- forward: one row per pair named by a reference that is new or changed since the cut, with the usable award ids of its
-- ROR / FundRef DOI references (as the reader did before the 10-16 switch, which gave every such reference an empty array).
-- Name-resolved references (exact, hierarchy, reviewed, twin, programme) add the link and no ids (#1303).
-- A new pair enters with those ids; a published pair has the ids it lacks appended.
CREATE OR REPLACE TABLE openalex.awards.crossref_work_funder_evidence
USING delta AS
WITH preserved AS (
 SELECT * FROM openalex.awards.crossref_work_funders
 UNION ALL
 SELECT m.* FROM openalex.awards.crossref_no_doi_main_cut m
 LEFT ANTI JOIN openalex.awards.crossref_work_funders p USING(work_id,funder_id)
),
forward AS (
 SELECT n.work_id,n.funder_id,
 array_sort(array_distinct(collect_list(CASE WHEN n.resolution_path IN ('ROR','FundRef DOI')
  AND openalex.common.is_usable_award_id(aid) THEN aid END))) award_ids
 FROM crossref_funder_name_candidates n
 LATERAL VIEW OUTER explode(n.award_ids) a AS aid
 WHERE n.hold_reason IS NULL
 GROUP BY n.work_id,n.funder_id
)
SELECT work_id,funder_id,CASE WHEN size(new_ids)>0 THEN concat(coalesce(award_ids,array()),new_ids) ELSE award_ids END award_ids,
 'preserved' disposition
FROM (SELECT p.work_id,p.funder_id,p.award_ids,array_sort(array_except(f.award_ids,coalesce(p.award_ids,array()))) new_ids
 FROM preserved p LEFT JOIN forward f USING(work_id,funder_id))
UNION ALL
SELECT n.work_id,n.funder_id,n.award_ids,'accepted' disposition
FROM forward n
LEFT ANTI JOIN preserved old USING(work_id,funder_id)


In [ ]:
%sql
-- Kyle's Crossref rule 1 (oxjobs #1890). A trap is a small funder that shares its name with a bigger one abroad. A link to a
-- trap is distrusted when no author is from the trap's country, an author is from a same-name twin's country, and no other
-- source names that funder on the paper. Traps: exact SQL mirror of notebooks/funders/crossref_trap_funders.csv (Kyle's
-- 129 hard and 21 shared-name traps, plus #1846's Arthritis NRF, whose "ANRF" deposits are India's Anusandhan NRF),
-- keyed on OpenAlex funder ids (re-keyed from Crossref registry IDs on 23 Nov 2026, oxjob #2373: the same 151 records),
-- checked by scripts/validate_funder_name_fixes.py. Countries: openalex.works.work_authorships.
CREATE OR REPLACE TEMP VIEW crossref_distrusted_links AS
WITH traps AS (
 SELECT f.funder_id,f.country_code,split(t.twin_countries,'[|]') twin_countries
 FROM VALUES
 (54609531,'AU|CN|CO|ES|US'),
 (3458447014,'US'),
 (3672381294,'BE'),
 (3745216989,'KR|NL'),
 (4320306227,'FR|IN|KR|SG|ZA'),
 (4320306941,'AU|GB'),
 (4320306962,'AU|DK|NL|NZ|SG'),
 (4320307330,'SE'),
 (4320308022,'DE'),
 (4320308088,'DE'),
 (4320308569,'US'),
 (4320309229,'GB'),
 (4320309450,'NZ'),
 (4320309580,'AU|CA|DE|GB|IE'),
 (4320309618,'BD|CN|ES|IL|KR|PK|TH|TW'),
 (4320309687,'CN|ES'),
 (4320309788,'AE|AU|KE|KR|NO|SG|US|ZA'),
 (4320310282,'CN'),
 (4320310459,'CH|GR|IN|SE'),
 (4320310681,'CN|ES'),
 (4320310773,'CA|JP|KR|NG'),
 (4320311039,'AR|BD|BN|CA|CL|CN|CZ|DZ|EG|ET|IL|IN|IQ|JP|KE|KR|LY|MY|SA|SD|SG|TH|TN|TR|TW'),
 (4320311419,'BF|BR|CN|DK|ES|IL|MX|MY|RU|SA|SG|SI|UG'),
 (4320311649,'AE|AR|BD|BN|CA|CL|CN|CZ|ET|IL|IN|JP|KE|KR|MY|SA|SG|TR|TW'),
 (4320311687,'AE|AR|BD|BN|CA|CL|CN|CZ|ET|IL|IN|JP|KE|KR|MY|SA|SG|TH|TR'),
 (4320311694,'ES'),
 (4320312014,'IT'),
 (4320312169,'EG|US'),
 (4320313387,'BE'),
 (4320313406,'AU|CA|CN'),
 (4320313482,'IN'),
 (4320313765,'US'),
 (4320313918,'AU|CA|DE|IE|US'),
 (4320313956,'CN|SE|SK'),
 (4320314062,'CA|EG|IT|KR|LK|LT|RO'),
 (4320315682,'CN|VN'),
 (4320315893,'EG|MO'),
 (4320316035,'CA|HK|IN|LK|NP'),
 (4320316371,'BR'),
 (4320317041,'FR|JP'),
 (4320317212,'BG'),
 (4320317988,'IT'),
 (4320318902,'AR|BR|CA|ES|FR|MY|SG|US'),
 (4320319387,'BR|CA|CN|ES|GB|PT|RO|US|UY'),
 (4320319573,'CN|GB|SG|TW'),
 (4320319853,'FR'),
 (4320319929,'AU|CN|CZ|GB|SE|TH|US'),
 (4320320052,'CA|PT'),
 (4320320138,'TR'),
 (4320320373,'GB|LK|NO|US'),
 (4320320381,'DE|US'),
 (4320320385,'SE|US'),
 (4320320505,'CN|FR'),
 (4320320671,'AE|AU|GB|KE|KR|NO|SG|US'),
 (4320320682,'IN'),
 (4320320691,'AU|IN|IT'),
 (4320320835,'AU|FR|SE|US'),
 (4320321147,'MY|SA'),
 (4320321408,'AE|AR|BD|BN|CA|CL|CN|CZ|ET|IL|IN|JP|KE|MY|SA|SG|TH|TR|TW'),
 (4320321655,'MO|VN'),
 (4320321701,'AE|BD|CN|CZ|IN|JP|KE|KR|MY|SA|SG|TH|TW'),
 (4320321842,'CZ'),
 (4320321934,'CN'),
 (4320322108,'BD|CN|ES|IL|PK|TH|TW|VN'),
 (4320322278,'NL|US'),
 (4320322357,'JP|US'),
 (4320322364,'FR'),
 (4320322429,'US|VN'),
 (4320322730,'AE|AR|BD|CA|CL|CN|IL|IN|KE|KR|MY|SA|SG|TH|TR|TW'),
 (4320322835,'NL'),
 (4320322986,'BE|US'),
 (4320323412,'CN'),
 (4320323706,'BD|CN|ES|KR|PK|TW|VN'),
 (4320323990,'DE'),
 (4320324195,'AU|GB|MU|QA|SL|US'),
 (4320324215,'BG|CA|FR|US'),
 (4320324279,'BD|CN|IL|KR|PK|TW|VN'),
 (4320324301,'CA|IR|TH|TW|US'),
 (4320324334,'AE|DZ|EG|LY|SD|TN'),
 (4320324450,'CH|DE|GB|HK|IN|IT|SE|US'),
 (4320324786,'BE|DE|GB|HK|IN|IT|SE|US'),
 (4320325299,'ES'),
 (4320325315,'CL'),
 (4320325366,'DE'),
 (4320325593,'CL'),
 (4320325614,'PT|US'),
 (4320325670,'EE|ES|RO'),
 (4320326283,'US'),
 (4320326444,'NL|PT|US'),
 (4320326574,'AR'),
 (4320326825,'IS'),
 (4320327261,'PL'),
 (4320327322,'AU|CA|GB|QA|SL|US'),
 (4320327529,'PK|US'),
 (4320327531,'AE|AU|GB|KE|KR|SG|US|ZA'),
 (4320328088,'MX'),
 (4320328470,'CA|US'),
 (4320328544,'SA|TR'),
 (4320328551,'CA|CN|FR|US'),
 (4320328747,'ID|IN|JP'),
 (4320329200,'US'),
 (4320330134,'GB'),
 (4320330374,'ID|NP|PT'),
 (4320330427,'GB|US'),
 (4320330647,'CN'),
 (4320330665,'FR'),
 (4320330717,'AU|CA|DE|US'),
 (4320330889,'AU|LK|NO|US'),
 (4320330949,'US'),
 (4320331015,'AU|CA|GB|MU|QA|US'),
 (4320331163,'AE|AR|BD|BN|CA|CL|CN|CZ|ET|IL|IN|JP|KR|MY|SA|SG|TH|TR|TW'),
 (4320331164,'TW|US'),
 (4320331439,'CA|JP|LK|PL|US'),
 (4320331609,'AT'),
 (4320332057,'FR'),
 (4320332238,'AU|CA|CN|CZ|TH'),
 (4320332623,'IT'),
 (4320332647,'FR|IT'),
 (4320332813,'US'),
 (4320332857,'IN'),
 (4320333169,'AU|GB|KE|KR|NO|SG|US|ZA'),
 (4320333333,'FR'),
 (4320333359,'CA|FR|LB'),
 (4320333368,'AU'),
 (4320333408,'FR|IE|US'),
 (4320333968,'CN'),
 (4320334321,'AU|CA|GB|MU|QA|SL'),
 (4320334796,'AU|US'),
 (4320335090,'CN|ES'),
 (4320335202,'AU|MD|US'),
 (4320335321,'CA|CO|ES|FR|US'),
 (4320335334,'IN|IT|ZA'),
 (4320335353,'AU|GB|NO|US'),
 (4320335542,'CL'),
 (4320336370,'GB|JP|LK|PL|RO|US'),
 (4320337157,'CA|CO|FR|MY|SG|US'),
 (4320337174,'AU|SA'),
 (4320337197,'ES|PT'),
 (4320337286,'CN'),
 (4320337305,'ES|PT'),
 (4320337504,'CN'),
 (4320337515,'AU|CA|CN|CZ|TH'),
 (4320338275,'AU|DK|EG|FR|IT|NO|NZ|VN|ZA'),
 (4585834687,'NO|US'),
 (4916976011,'AU|IN|PH|ZA'),
 (5477361653,'CN'),
 (6112533845,'CA|US'),
 (6916241136,'IL'),
 (7099196522,'BD|DK|KR|TW|ZM'),
 (7400742617,'FR'),
 (7554583698,'CN|IN|JP')
 t(distrusted_funder_id,twin_countries)
 JOIN openalex.funders.funders f ON f.funder_id=t.distrusted_funder_id AND f.merge_into_id IS NULL
),
trap_links AS (
 SELECT e.work_id,e.funder_id,t.funder_id trap_id
 FROM openalex.awards.crossref_work_funder_evidence e
 JOIN openalex.funders.funders f ON f.funder_id=e.funder_id
 JOIN traps t ON t.funder_id=coalesce(f.merge_into_id,f.funder_id)
 JOIN (SELECT work_id,flatten(transform(authorships,a->a.countries)) countries FROM openalex.works.work_authorships) w ON w.work_id=e.work_id
 WHERE NOT array_contains(w.countries,t.country_code) AND arrays_overlap(w.countries,t.twin_countries)
),
named_elsewhere AS (
 SELECT work_id,funder_id FROM openalex.awards.europepmc_work_funders
 UNION ALL SELECT work_id,funder_id FROM openalex.awards.datacite_work_funders
 UNION ALL SELECT work_id,funder_id FROM openalex.awards.funder_reported_work_funders
 UNION ALL SELECT work_id,funder_id FROM openalex.awards.kaken_work_funders
 UNION ALL SELECT work_id,funder_id FROM openalex.awards.anr_work_funders
 UNION ALL SELECT work_id,funder_id FROM openalex.awards.nwo_work_funders
 UNION ALL SELECT work_id,funder_id FROM openalex.awards.hakai_work_funders
 UNION ALL SELECT work_id,funder_id FROM openalex.awards.gtr_awards WHERE work_id IS NOT NULL
 UNION ALL SELECT work_id,try_cast(replace(funder_id,'https://openalex.org/F','') AS BIGINT) FROM openalex.works.fulltext_work_funders
)
SELECT DISTINCT l.work_id,l.funder_id FROM trap_links l
LEFT ANTI JOIN named_elsewhere o ON o.work_id=l.work_id AND o.funder_id=l.trap_id


In [ ]:
%sql
-- Distrusted pairs leave the junction below, with their award ids. Rebuilt from the inputs every run.
MERGE INTO openalex.awards.crossref_work_funder_evidence e
USING crossref_distrusted_links d ON e.work_id=d.work_id AND e.funder_id=d.funder_id
WHEN MATCHED THEN UPDATE SET e.disposition='distrusted'


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.crossref_work_funders
USING delta AS
SELECT work_id,funder_id,award_ids FROM openalex.awards.crossref_work_funder_evidence
WHERE disposition<>'distrusted'


### Sanity checks

In [ ]:
%sql
-- Row counts and uniqueness
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id)) AS distinct_keys,
  COUNT(DISTINCT work_id) AS distinct_works,
  COUNT(DISTINCT funder_id) AS distinct_funders,
  COUNT(CASE WHEN SIZE(award_ids) > 0 THEN 1 END) AS rows_with_awards
FROM openalex.awards.crossref_work_funders

In [ ]:
%sql
-- Verify composite key uniqueness (should return 0)
SELECT COUNT(*) AS duplicate_keys
FROM (
  SELECT work_id, funder_id, COUNT(*) AS n
  FROM openalex.awards.crossref_work_funders
  GROUP BY work_id, funder_id
  HAVING n > 1
)

### Insert awards into `openalex_awards_raw`

Follows the same pattern as `CreateBackfillAwards`: explode award_ids, generate xxhash64-based IDs, insert with provenance and priority.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW crossref_work_funder_awards AS
WITH exploded AS (
  SELECT DISTINCT
    cwf.funder_id,
    LOWER(award_id) AS normalized_award_id,
    award_id AS funder_award_id
  FROM openalex.awards.crossref_work_funders cwf
  LATERAL VIEW EXPLODE(award_ids) AS award_id
  WHERE SIZE(award_ids) > 0
),
-- #690 verdict-consuming guard: ids scored 'garbage' for a configured funder
-- with NO salvage rescue (decorated own-id / multi-id split / wrong-funder
-- detection) do not mint award entities. Fail-open by design: ids with no
-- guard row (new since the last AwardNormKey scoring run, or at unconfigured
-- funders -> verdict 'unscored' -> decision 'mint') mint exactly as before.
-- Ordering: openalex.awards.award_id_guard (AwardNormKey.sql) must exist
-- before this leg first runs with the guard in place.
guarded AS (
  SELECT e.funder_id, e.normalized_award_id, e.funder_award_id
  FROM exploded e
  LEFT JOIN openalex.awards.award_id_guard g
    ON g.funder_id = e.funder_id AND g.funder_award_id = e.funder_award_id
  WHERE COALESCE(g.decision, 'mint') <> 'suppress'
),
funders AS (
  SELECT funder_id, display_name, ror_id, doi
  FROM openalex.funders.funders
)
SELECT
  ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000 AS id,
  CAST(NULL AS STRING) AS display_name,
  CAST(NULL AS STRING) AS description,
  f.funder_id,
  e.funder_award_id,
  CAST(NULL AS DOUBLE) AS amount,
  CAST(NULL AS STRING) AS currency,
  STRUCT(
    CONCAT('https://openalex.org/F', f.funder_id) AS id,
    f.display_name,
    f.ror_id,
    f.doi
  ) AS funder,
  CAST(NULL AS STRING) AS funding_type,
  CAST(NULL AS STRING) AS funder_scheme,
  'crossref_work_funders' AS provenance,
  CAST(NULL AS DATE) AS start_date,
  CAST(NULL AS DATE) AS end_date,
  CAST(NULL AS INT) AS start_year,
  CAST(NULL AS INT) AS end_year,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS lead_investigator,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS co_lead_investigator,
  CAST(NULL AS ARRAY<STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >>) AS investigators,
  CAST(NULL AS STRING) AS landing_page_url,
  openalex.common.extract_grant_doi(e.funder_award_id) AS doi,  -- oxjobs #172: lift grant DOIs
  CONCAT('https://api.openalex.org/works?filter=awards.id:G', ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000) AS works_api_url,
  CURRENT_TIMESTAMP() AS created_date,
  CURRENT_TIMESTAMP() AS updated_date
FROM guarded e
JOIN funders f ON f.funder_id = e.funder_id
-- #690 ticket 7682: an ERC call identifier (ERC-2021-ADG, ERC‐2015‐AdG with Unicode hyphens, ERC 2019 StG ...) names
-- an ERC call, never a grant. Under an EU programme funder (European Commission, European Research Council, FP7, H2020,
-- Horizon Europe and their programme children: 86 funders, whose grants are the 6- or 9-digit agreement numbers of the
-- erc and cordis registries, none of which has this shape) it mints no award; works keep the funder and their real
-- grant numbers. Other funders are untouched (Kyle 2026-07-28 rule 4: the guard runs only where a registry tests it).
WHERE NOT (coalesce(f.display_name, '') RLIKE '^(European Commission|European Research Council|Seventh Framework Programme|Horizon 2020( Framework Programme)?|(FP7|H2020|HORIZON EUROPE) .+)$'
  AND regexp_replace(upper(trim(regexp_replace(e.funder_award_id, '[\\u2010-\\u2015\\u2212\\uFE58\\uFE63\\uFF0D]', '-'))), '\\s+', ' ')
      RLIKE '^ERC[-_ ]?(19|20)[0-9]{2}[-_ ]?(STG|COG|ADG|SYG|POC[0-9]?|POC[-_ ]?(LS|SM)[0-9]?)$')

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'crossref_work_funders' AND priority IN (0, 2)

In [ ]:
%sql
INSERT INTO openalex.awards.openalex_awards_raw
SELECT *, 0 AS priority
FROM crossref_work_funder_awards

In [ ]:
%sql
-- #690 guard telemetry: how many junction award ids the checkpoint suppressed
-- this run (verdict 'garbage', no salvage rescue). Expect a stable-ish number;
-- a jump means a scoring or registry regression upstream.
SELECT COUNT(*) AS suppressed_ids, COUNT(DISTINCT j.funder_id) AS funders_hit
FROM (SELECT DISTINCT funder_id, aid AS funder_award_id
      FROM openalex.awards.crossref_work_funders LATERAL VIEW EXPLODE(award_ids) AS aid) j
JOIN openalex.awards.award_id_guard g
  ON g.funder_id = j.funder_id AND g.funder_award_id = j.funder_award_id
WHERE g.decision = 'suppress'

In [ ]:
%sql
SELECT assert_true(count(*)=0,'Crossref: preserved pair or award array changed')
FROM openalex.awards.crossref_work_funder_evidence e
LEFT JOIN openalex.awards.crossref_work_funders j USING(work_id,funder_id)
WHERE e.disposition='preserved' AND (j.work_id IS NULL OR NOT(e.award_ids<=>j.award_ids))
